In [0]:
%sql
--create a table for silver
CREATE TABLE IF NOT EXISTS silver.day14_customers (
    CustomerId INT,
    CustomerName STRING,
    City STRING,
    Age INT,
    UpdatedAt TIMESTAMP
)
USING DELTA;
--create a table for quarantine
CREATE TABLE IF NOT EXISTS silver.day14_customer_quarantine (
    CustomerId INT,
    CustomerName STRING,
    City STRING,
    Age INT,
    UpdatedAt STRING,
    dq_reason STRING,
    quarantined_at TIMESTAMP
)
USING DELTA;

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

customer_schema = StructType([
    StructField("CustomerId", IntegerType(), True),
    StructField("CustomerName", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("UpdatedAt", StringType(), True)
])
bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(customer_schema)
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/bronze/schema/day14_customers/")
    .load("/Volumes/workspace/bronze/day14_customer_files/")
)
bronze_query = (
    bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/workspace/bronze/checkpoints/day14_bronze/")
    .trigger(availableNow=True)
    .toTable("bronze.day14_customers")
)
bronze_query.awaitTermination()

In [0]:
%sql
SELECT *
FROM bronze.day14_customers
ORDER BY CustomerId;

CustomerId,CustomerName,City,Age,UpdatedAt,_rescued_data
105,Meena,Madurai,29,2026-08-22 09:00:00,null
106,null,Chennai,32,2026-08-22 09:05:00,null
107,Ravi,Coimbatore,-5,2026-08-22 09:10:00,null
108,Divya,null,27,2026-08-22 09:15:00,null
109,Karthik,Salem,35,2026-08-22 09:20:00,null


In [0]:
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from pyspark.sql.functions import (
        col,
        to_timestamp,
        when,
        lit,
        current_timestamp,
        concat_ws,
        row_number
    )

# Reading Bronze as streaming
silver_stream = (
    spark.readStream
    .table("bronze.day14_customers")
)

def process_customer_quality(batch_df, batch_id):
    # Parse UpdatedAt
    batch_df = batch_df.withColumn(
        "UpdatedAtParsed",
        to_timestamp(col("UpdatedAt"))
    )
    #added dq_reason column
    batch_df = batch_df.withColumn(
        "dq_reason",
        concat_ws(
            "; ",
            when(col("CustomerId").isNull(), lit("CustomerId is NULL")),
            when(col("CustomerName").isNull(), lit("CustomerName is NULL")),
            when(col("City").isNull(), lit("City is NULL")),
            when(col("Age").isNull(), lit("Age is NULL")),
            when(col("Age") <= 0, lit("Age must be greater than 0")),
            when(col("UpdatedAtParsed").isNull(), lit("UpdatedAt is invalid"))
        )
    )
    #validted condition
    valid_condition = (
    col("CustomerId").isNotNull()
    & col("CustomerName").isNotNull()
    & col("City").isNotNull()
    & col("Age").isNotNull()
    & (col("Age") > 0)
    & col("UpdatedAtParsed").isNotNull()
    )

    batch_df = batch_df.withColumn(
    "is_valid",
    valid_condition
    )

    valid_df = (
    batch_df
    .filter(col("is_valid"))
    )

    invalid_df = (
    batch_df
    .filter(~col("is_valid"))
    )

    invalid_df = (
    invalid_df
    .withColumn("quarantined_at", current_timestamp())
    .select(
        "CustomerId",
        "CustomerName",
        "City",
        "Age",
        "UpdatedAt",
        "dq_reason",
        "quarantined_at"
    )
    )
    # Write to quarantine table
    invalid_df.write.mode("append").saveAsTable("silver.day14_customer_quarantine")

    valid_df = (
    valid_df
    .withColumn("UpdatedAt", col("UpdatedAtParsed"))
    .drop("UpdatedAtParsed", "is_valid", "dq_reason")
    )

    window_spec = (
        Window
        .partitionBy("CustomerId")
        .orderBy(col("UpdatedAt").desc())
    )

    latest_valid = (
        valid_df
        .withColumn("row_num", row_number().over(window_spec))
        .filter(col("row_num") == 1)
        .drop("row_num")
    )

    silver_table = DeltaTable.forName(
    batch_df.sparkSession,
    "silver.day14_customers"
    )
    #merge with deduplication check
    (
        silver_table.alias("target")
        .merge(
            latest_valid.alias("source"),
            "target.CustomerId = source.CustomerId"
        )
        .whenMatchedUpdate(
            condition="source.UpdatedAt > target.UpdatedAt",
            set={
                "CustomerName": "source.CustomerName",
                "City": "source.City",
                "Age": "source.Age",
                "UpdatedAt": "source.UpdatedAt"
            }
        )
        .whenNotMatchedInsert(
            values={
                "CustomerId": "source.CustomerId",
                "CustomerName": "source.CustomerName",
                "City": "source.City",
                "Age": "source.Age",
                "UpdatedAt": "source.UpdatedAt"
            }
        )
        .execute()
    )


quality_query = (
    silver_stream.writeStream
    .foreachBatch(process_customer_quality)
    .option("checkpointLocation", "/Volumes/workspace/silver/checkpoints/day14_quality/")
    .trigger(availableNow=True)
    .start()
)
quality_query.awaitTermination()

In [0]:
%sql
SELECT *
FROM silver.day14_customers
ORDER BY CustomerId;

CustomerId,CustomerName,City,Age,UpdatedAt
105,Meena,Madurai,29,2026-08-22T09:00:00.000Z
109,Karthik,Salem,35,2026-08-22T09:20:00.000Z


In [0]:
%sql
SELECT *
FROM silver.day14_customer_quarantine
ORDER BY CustomerId;

CustomerId,CustomerName,City,Age,UpdatedAt,dq_reason,quarantined_at
106,null,Chennai,32,2026-08-22 09:05:00,CustomerName is NULL,2026-08-22T10:50:56.901Z
107,Ravi,Coimbatore,-5,2026-08-22 09:10:00,Age must be greater than 0,2026-08-22T10:50:56.901Z
108,Divya,null,27,2026-08-22 09:15:00,City is NULL,2026-08-22T10:50:56.901Z
